In [1]:
# Import required libraries
import numpy as np
import scipy
import scipy.linalg as sla
import matplotlib.pyplot as plt

In [2]:
# Read the required file | Roll No: 23b2157
roll_no = "23b2157.npz"
with np.load(roll_no) as system:
    u = system["x"]
    f = system["b"]
    K = scipy.sparse.coo_matrix(
        (system["A_values"], list(system["A_indices"])),
        shape=(u.size, u.size)
    )

In [3]:
# Check K * u = f
if(np.allclose(K.dot(u), f)):
    print("Everything is correct!")
else:
    print("There is something wrong!")

print("K shape:",K.shape)
print("u shape:",u.shape)
print("f shape:",f.shape)

Everything is correct!
K shape: (1494, 1494)
u shape: (1494,)
f shape: (1494,)


In [4]:
# LU factorisation for sparse matrix using scipy
lu = scipy.sparse.linalg.splu(K)
L = lu.L
U = lu.U

/var/folders/vd/jt4zk_ks2j57r_s_8ts2880c0000gn/T/ipykernel_10709/2999175385.py:2: SparseEfficiencyWarning: splu converted its input to CSC format
  lu = scipy.sparse.linalg.splu(K)


In [5]:
L_csc = L.tocsc()
U_csr = U.tocsr()
col_nnz_L = np.diff(L_csc.indptr)   # nnz per column of L
row_nnz_U = np.diff(U_csr.indptr)   # nnz per row of U
flops_factor_est = 2 * np.sum(col_nnz_L * row_nnz_U)
print("Factorization FlOps:", flops_factor_est)

Factorization FlOps: 35182542


In [6]:
# Record the factorisation time using timeit
lu_factor_time_sparse = %timeit -o scipy.sparse.linalg.splu(K)

<magic-timeit>:1: SparseEfficiencyWarning: splu converted its input to CSC format


9.47 ms ± 638 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
print("Average Time:",lu_factor_time_sparse)
print("Minimum Time (No noise):",lu_factor_time_sparse.best)

Average Time: 9.47 ms ± 638 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Minimum Time (No noise): 0.008944632920000002


In [8]:
# Solve the factorised matrices using sparse forward and back sub
u_solved_sparse = lu.solve(f)
residual_sparse = np.linalg.norm(K.dot(u_solved_sparse) - f)

In [9]:
print("Residual:",residual_sparse)

Residual: 7.182850177464843e-09


In [10]:
# Record the time required to solve using sparse forward and back sub
lu_solve_time_sparse = %timeit -o lu.solve(f)

165 µs ± 9.41 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [11]:
print("Average Time:",lu_solve_time_sparse)
print("Minimum Time:",lu_solve_time_sparse.best)
print("FlOps:",2*(L.nnz+U.nnz))
print("GFlOpS:",2*(L.nnz+U.nnz)/(lu_solve_time_sparse.average*10**9)*1.0)

Average Time: 165 µs ± 9.41 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Minimum Time: 0.0001540413832999999
FlOps: 550868
GFlOpS: 3.3349648794094877


In [12]:
K_dense = K.toarray()

lu_dense_factor_time = %timeit -o sla.lu_factor(K_dense)

29.4 ms ± 2.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
# Dense FLOPs — theoretical formula applies directly here (no fill-in trick needed)
n = K_dense.shape[0]
flops_factor_dense = (2/3) * n**3
print("Factorization FlOps:", flops_factor_dense)

Factorization FlOps: 2223107856.0


In [14]:
lu_piv = sla.lu_factor(K_dense)   # returns (LU combined, piv)
print("Average Time:", lu_dense_factor_time)
print("Minimum Time (No noise):", lu_dense_factor_time.best)

Average Time: 29.4 ms ± 2.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
Minimum Time (No noise): 0.027776454199999988


In [15]:
# Solve using dense forward/back sub
u_solved_dense = sla.lu_solve(lu_piv, f)
residual_dense = np.linalg.norm(K_dense @ u_solved_dense - f)
print("Residual:", residual_dense)

Residual: 6.500720530090227e-09


In [16]:
# Record the time required to solve using traditional forward and back sub
lu_dense_solve_time = %timeit -o sla.lu_solve(lu_piv, f)

484 µs ± 38.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
print("Average Time:", lu_dense_solve_time)
print("Minimum Time:", lu_dense_solve_time.best)
print("FlOps:", 2 * n**2)   # dense forward/back sub is O(n²), not O(nnz)
print("GFlOpS:", 2*n**2/(lu_dense_solve_time.average*10**9))

Average Time: 484 µs ± 38.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Minimum Time: 0.00045955045799999896
FlOps: 4464072
GFlOpS: 9.22470066262729


In [18]:
gflops_sparse_factor = flops_factor_est / (lu_factor_time_sparse.average * 1e9)
gflops_sparse_solve  = 2*(L.nnz+U.nnz) / (lu_solve_time_sparse.average * 1e9)
gflops_dense_factor  = flops_factor_dense / (lu_dense_factor_time.average * 1e9)
gflops_dense_solve   = 2*n**2 / (lu_dense_solve_time.average * 1e9)

print("Sparse factor GFlOp/s:", gflops_sparse_factor)
print("Sparse solve  GFlOp/s:", gflops_sparse_solve)
print("Dense  factor GFlOp/s:", gflops_dense_factor)
print("Dense  solve  GFlOp/s:", gflops_dense_solve)

Sparse factor GFlOp/s: 3.714436870959032
Sparse solve  GFlOp/s: 3.3349648794094877
Dense  factor GFlOp/s: 75.50964468841634
Dense  solve  GFlOp/s: 9.22470066262729


In [19]:
# Memory for sparse
def memory_bytes(mat):
    if mat.format == 'coo':
        return mat.data.nbytes + mat.row.nbytes + mat.col.nbytes
    elif mat.format in ('csc', 'csr'):
        return mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
    else:
        return mat.tocoo().data.nbytes * 3  # fallback estimate

print("Storage for Sparse:\nK (COO):", memory_bytes(K) / 1024, "KB")
print("L (CSC):", memory_bytes(L) / 1024, "KB")
print("U (CSC):", memory_bytes(U) / 1024, "KB")

# Memory — dense storage, not nnz-based
mem_dense_bytes = K_dense.nbytes   # n² × 8 bytes for float64
print("Dense storage (MB):", mem_dense_bytes / 1024**2)

Storage for Sparse:
K (COO): 250.96875 KB
L (CSC): 1619.7109375 KB
U (CSC): 1619.7109375 KB
Dense storage (MB): 17.029083251953125


In [20]:
np.show_config()

{
  "Compilers": {
    "c": {
      "name": "clang",
      "linker": "ld64",
      "version": "15.0.0",
      "commands": "cc"
    },
    "cython": {
      "name": "cython",
      "linker": "cython",
      "version": "3.0.11",
      "commands": "cython"
    },
    "c++": {
      "name": "clang",
      "linker": "ld64",
      "version": "15.0.0",
      "commands": "c++"
    }
  },
  "Machine Information": {
    "host": {
      "cpu": "aarch64",
      "family": "aarch64",
      "endian": "little",
      "system": "darwin"
    },
    "build": {
      "cpu": "aarch64",
      "family": "aarch64",
      "endian": "little",
      "system": "darwin"
    }
  },
  "Build Dependencies": {
    "blas": {
      "name": "accelerate",
      "found": true,
      "version": "unknown",
      "detection method": "system",
      "include directory": "unknown",
      "lib directory": "unknown",
      "openblas configuration": "unknown",
      "pc file directory": "unknown"
    },
    "lapack": {
      "name

/Users/rishabhjalgaonkar/Documents/Mechanical_Engineering/Semester 7/ME789/.venv/lib/python3.9/site-packages/numpy/__config__.py:155: UserWarning: Install `pyyaml` for better output
  warnings.warn("Install `pyyaml` for better output", stacklevel=1)
